# NB4 — Visualisation (Google Colab version)

**COS700 · Impact of Data Anonymisation on the Statistical Utility of Cybersecurity Datasets**

**Phases 5–7 deliverables. Google Colab version.**  
Inputs: `COS700/results/` and `COS700/data/analysis.parquet` on your Google Drive  
Outputs: `COS700/figures/experiment/` on your Google Drive

## Before running
1. NB1, NB2, and NB3 (Colab versions) must have completed successfully.
2. Set `DRIVE_ROOT` in the setup cell below if your folder path differs.
3. Run all cells top-to-bottom.


In [ ]:
from pathlib import Path

# ── Install dependencies (only needed once per Colab session) ─────────────────
import subprocess, sys
pkgs = ['pyarrow']
for pkg in pkgs:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

# ── Mount Google Drive ────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

# ── Set your Drive root folder ────────────────────────────────────────────────
DRIVE_ROOT = '/content/drive/MyDrive/COS700'

print(f'Drive mounted. Using root: {DRIVE_ROOT}')


# NB4 — Visualisation

**COS700 · Impact of Data Anonymisation on the Statistical Utility of Cybersecurity Datasets**

**Phases 5–7** deliverables.  
Inputs: `results/metrics.csv`, `results/spearman.csv`, `results/gini_importance.csv`, `results/feature_meta.csv`  
Outputs: `figures/experiment/`

| Figure | File |
|---|---|
| JSD heatmaps (noise, suppression, DP) | `heatmap_jsd_*.png` |
| W1 heatmaps | `heatmap_w1_*.png` |
| KL heatmaps | `heatmap_kl_*.png` |
| DP ε-curve (JSD + W1 observed/predicted) | `dp_epsilon_curve.png` |
| Spearman ρ summary | `spearman_rho.png` |
| Category-level JSD (per arm) | `category_jsd_*.png` |
| DP clipping-loss decomposition | `dp_clip_decomposition.png` |

**Run all cells top-to-bottom.**

## 0. Imports & setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# ── Paths ──────────────────────────────────────────────────────────────────────
ROOT      = Path(DRIVE_ROOT)
RESULTS   = ROOT / 'results'
FIG_DIR   = ROOT / 'figures' / 'experiment'
FIG_DIR.mkdir(parents=True, exist_ok=True)

# ── Plot style ────────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi':      150,
    'font.size':       9,
    'axes.titlesize':  10,
    'axes.labelsize':  9,
    'xtick.labelsize': 8,
    'ytick.labelsize': 7,
    'figure.facecolor': 'white',
    'axes.facecolor':   'white',
})

DPI = 200  # save DPI

print('Imports OK. Figures will be saved to:', FIG_DIR)

## 1. Load inputs

In [ ]:
metrics  = pd.read_csv(RESULTS / 'metrics.csv')
spearman = pd.read_csv(RESULTS / 'spearman.csv')
gini     = pd.read_csv(RESULTS / 'gini_importance.csv')
meta     = pd.read_csv(RESULTS / 'feature_meta.csv')

# Feature order: Gini importance descending (for heatmap rows)
FEAT_ORDER = gini.sort_values('importance', ascending=False)['feature'].tolist()
d = len(FEAT_ORDER)

DP_EPSILONS = [0.01, 0.1, 0.5, 1.0, 5.0, 10.0]
CATEGORIES  = ['Packet lengths', 'Flow timing', 'Packet counts',
                'Rates & volumes', 'TCP flags', 'Headers & window']

# Category colours (consistent across all figures)
CAT_COLORS = {
    'Packet lengths':  '#4C72B0',
    'Flow timing':     '#DD8452',
    'Packet counts':   '#55A868',
    'Rates & volumes': '#C44E52',
    'TCP flags':       '#8172B3',
    'Headers & window':'#937860',
}

print(f'Loaded metrics ({len(metrics)} rows), spearman ({len(spearman)} rows), '
      f'gini ({len(gini)} features)')

## 2. Helper: saturation mask

Pre-compute which DP (feature × ε) cells are in the plateau (bin-width ratio > 100).
Used to annotate the DP heatmaps.

In [ ]:
analysis = pd.read_parquet(ROOT / 'data' / 'analysis.parquet')
feature_cols = [c for c in analysis.columns if c != 'Label']

N_BINS = 100
sat = {}  # sat[(feat, eps)] = True if plateau
for feat in feature_cols:
    vals = analysis[feat].to_numpy(dtype=float)
    w_j  = max((vals.max() - vals.min()) / N_BINS, 1e-10)
    df_j = float(meta[meta['feature'] == feat]['delta_f'].values[0])
    for eps in DP_EPSILONS:
        sat[(feat, eps)] = (df_j / (eps * w_j)) > 100

plateau_count = sum(sat.values())
print(f'Plateau cells: {plateau_count} / {len(feature_cols) * len(DP_EPSILONS)} '
      f'({plateau_count / (len(feature_cols)*len(DP_EPSILONS))*100:.0f}%)')

## 3. Heatmap helper

In [ ]:
def make_heatmap(pivot: pd.DataFrame,
                 title: str,
                 cmap: str,
                 vmin: float,
                 vmax: float,
                 savepath: Path,
                 annot_plateau: pd.DataFrame = None,
                 annot_saturated: bool = False,
                 cbar_label: str = '') -> None:
    """Draw a feature × level heatmap.
    annot_plateau: boolean DataFrame same shape as pivot; True cells get 'P' marker.
    annot_saturated: if True, mark cells at KL saturation (≈23 nats) with '≈23'.
    """
    nrows, ncols = pivot.shape
    fig_h = max(4, nrows * 0.22)
    fig, ax = plt.subplots(figsize=(max(4, ncols * 1.4), fig_h))

    # Add category colour bar on the left
    feat_cats = [meta[meta['feature'] == f]['category'].values[0]
                 if f in meta['feature'].values else 'Unknown'
                 for f in pivot.index]
    cat_colors_list = [CAT_COLORS.get(c, '#999999') for c in feat_cats]

    sns.heatmap(
        pivot,
        ax=ax,
        cmap=cmap,
        vmin=vmin, vmax=vmax,
        linewidths=0.2, linecolor='#dddddd',
        cbar_kws={'label': cbar_label, 'shrink': 0.6},
        xticklabels=True, yticklabels=True,
    )

    # Colour-code the y-tick labels by category
    for ytick, colour in zip(ax.get_yticklabels(), cat_colors_list):
        ytick.set_color(colour)

    # Annotate plateau cells with 'P'
    if annot_plateau is not None:
        for i, feat in enumerate(pivot.index):
            for j, col in enumerate(pivot.columns):
                if annot_plateau.loc[feat, col]:
                    ax.text(j + 0.5, i + 0.5, 'P',
                            ha='center', va='center',
                            fontsize=5, color='white', alpha=0.7)

    # Annotate saturated KL cells
    if annot_saturated:
        for i, feat in enumerate(pivot.index):
            for j, col in enumerate(pivot.columns):
                if pivot.loc[feat, col] >= 20:
                    ax.text(j + 0.5, i + 0.5, '≈23',
                            ha='center', va='center',
                            fontsize=4.5, color='white', alpha=0.8)

    # Category legend
    from matplotlib.patches import Patch
    handles = [Patch(color=c, label=cat)
               for cat, c in CAT_COLORS.items()
               if cat in feat_cats]
    ax.legend(handles=handles, loc='upper left',
              bbox_to_anchor=(1.12, 1), fontsize=7,
              title='Category', title_fontsize=7,
              frameon=False)

    ax.set_title(title, pad=8)
    ax.set_ylabel('Feature (Gini ↓)', labelpad=4)
    ax.set_xlabel('')
    plt.tight_layout()
    fig.savefig(savepath, dpi=DPI, bbox_inches='tight')
    plt.show()
    print(f'  Saved: {savepath.name}')

print('Heatmap helper defined.')

## 4. JSD heatmaps

In [ ]:
# ── 4a: Noise ─────────────────────────────────────────────────────────────────
noise_df = metrics[metrics['technique'] == 'noise']
pivot = (
    noise_df.pivot(index='feature', columns='level', values='jsd')
    .reindex(FEAT_ORDER)
    .rename(columns={0.1: 'α=0.1', 0.5: 'α=0.5'})
)
make_heatmap(pivot, 'JSD — Noise perturbation', 'YlOrRd', 0, 1,
             FIG_DIR / 'heatmap_jsd_noise.png', cbar_label='JSD')

# ── 4b: Suppression ───────────────────────────────────────────────────────────
supp_df = metrics[metrics['technique'] == 'suppression']
pivot = (
    supp_df.pivot(index='feature', columns='level', values='jsd')
    .reindex(FEAT_ORDER)
    .rename(columns={0.25: '25%', 0.5: '50%', 0.75: '75%'})
)
make_heatmap(pivot, 'JSD — Suppression (mean-imputation)', 'YlOrRd', 0, 1,
             FIG_DIR / 'heatmap_jsd_suppression.png', cbar_label='JSD')

# ── 4c: DP ────────────────────────────────────────────────────────────────────
dp_df   = metrics[metrics['technique'] == 'dp']
clip_df = metrics[metrics['technique'] == 'dp_clip']

dp_pivot = (
    dp_df.pivot(index='feature', columns='level', values='jsd')
    .reindex(FEAT_ORDER)
)
clip_col = clip_df.set_index('feature')['jsd'].reindex(FEAT_ORDER)
dp_pivot.insert(0, 'clip\n(ε→∞)', clip_col.values)
dp_pivot.columns = ['clip\n(ε→∞)'] + [f'ε={e}' for e in DP_EPSILONS]

# Plateau mask
plateau_mask = pd.DataFrame(False, index=FEAT_ORDER,
                             columns=dp_pivot.columns)
for feat in FEAT_ORDER:
    for eps in DP_EPSILONS:
        col = f'ε={eps}'
        if col in plateau_mask.columns:
            plateau_mask.loc[feat, col] = sat.get((feat, eps), False)

make_heatmap(dp_pivot,
             'JSD — Differential privacy (per-value ε-local DP)\n'
             'P = JSD/KL plateau (bin-width ratio > 100)',
             'YlOrRd', 0, 1,
             FIG_DIR / 'heatmap_jsd_dp.png',
             annot_plateau=plateau_mask,
             cbar_label='JSD')

## 5. W1 heatmaps

In [ ]:
# Noise W1
pivot = (
    metrics[metrics['technique'] == 'noise']
    .pivot(index='feature', columns='level', values='wasserstein_norm')
    .reindex(FEAT_ORDER)
    .rename(columns={0.1: 'α=0.1', 0.5: 'α=0.5'})
)
vmax_w1_noise = float(pivot.values.max())
make_heatmap(pivot, 'W1 (σ-normalised) — Noise perturbation',
             'PuBu', 0, vmax_w1_noise,
             FIG_DIR / 'heatmap_w1_noise.png', cbar_label='W1 / σ')

# Suppression W1
pivot = (
    metrics[metrics['technique'] == 'suppression']
    .pivot(index='feature', columns='level', values='wasserstein_norm')
    .reindex(FEAT_ORDER)
    .rename(columns={0.25: '25%', 0.5: '50%', 0.75: '75%'})
)
make_heatmap(pivot, 'W1 (σ-normalised) — Suppression',
             'PuBu', 0, float(pivot.values.max()),
             FIG_DIR / 'heatmap_w1_suppression.png', cbar_label='W1 / σ')

# DP W1 — observed + predicted side by side
dp_w1_obs  = dp_df.pivot(index='feature', columns='level', values='wasserstein_norm').reindex(FEAT_ORDER)
dp_w1_pred = dp_df.pivot(index='feature', columns='level', values='w1_predicted').reindex(FEAT_ORDER)
dp_w1_obs.columns  = [f'ε={e}'  for e in DP_EPSILONS]
dp_w1_pred.columns = [f'ε={e} (pred)' for e in DP_EPSILONS]

# Use log scale for DP W1 (spans many orders of magnitude)
dp_w1_obs_log  = np.log10(dp_w1_obs.clip(lower=1e-3))
dp_w1_pred_log = np.log10(dp_w1_pred.clip(lower=1e-3))

nrows = len(FEAT_ORDER)
fig, axes = plt.subplots(1, 2, figsize=(18, max(5, nrows * 0.22)),
                         sharey=True)
vmin_log = min(dp_w1_obs_log.values.min(), dp_w1_pred_log.values.min())
vmax_log = max(dp_w1_obs_log.values.max(), dp_w1_pred_log.values.max())

for ax, data, subtitle in zip(axes,
                               [dp_w1_obs_log, dp_w1_pred_log],
                               ['Observed', 'Predicted asymptote Δf/(ε·σ)']):
    sns.heatmap(data, ax=ax, cmap='PuBu',
                vmin=vmin_log, vmax=vmax_log,
                linewidths=0.2, linecolor='#dddddd',
                cbar_kws={'label': 'log₁₀(W1/σ)', 'shrink': 0.5},
                xticklabels=True, yticklabels=True)
    ax.set_title(f'W1 (σ-normalised, log₁₀) — DP — {subtitle}')
    ax.set_xlabel('')
    ax.set_ylabel('Feature (Gini ↓)' if ax is axes[0] else '')

fig.suptitle('DP W1: observed ≈ predicted → W1 is design-predictable (REV-W1primary)',
             fontsize=9, y=1.01)
plt.tight_layout()
path = FIG_DIR / 'heatmap_w1_dp.png'
fig.savefig(path, dpi=DPI, bbox_inches='tight')
plt.show()
print(f'  Saved: {path.name}')

## 6. KL heatmaps

In [ ]:
KL_SAT = 23.0  # λ-smoothed saturation ceiling ≈ ln(10^10) nats

# Noise KL
pivot = (
    metrics[metrics['technique'] == 'noise']
    .pivot(index='feature', columns='level', values='kl')
    .reindex(FEAT_ORDER)
    .rename(columns={0.1: 'α=0.1', 0.5: 'α=0.5'})
)
make_heatmap(pivot, 'KL D(P‖Q) (nats) — Noise perturbation\n(saturated ≈ 23 nats)',
             'Greens', 0, KL_SAT,
             FIG_DIR / 'heatmap_kl_noise.png',
             annot_saturated=True, cbar_label='KL (nats)')

# Suppression KL
pivot = (
    metrics[metrics['technique'] == 'suppression']
    .pivot(index='feature', columns='level', values='kl')
    .reindex(FEAT_ORDER)
    .rename(columns={0.25: '25%', 0.5: '50%', 0.75: '75%'})
)
make_heatmap(pivot, 'KL D(P‖Q) (nats) — Suppression\n(≈23 = saturated; sparse features → point mass)',
             'Greens', 0, KL_SAT,
             FIG_DIR / 'heatmap_kl_suppression.png',
             annot_saturated=True, cbar_label='KL (nats)')

# DP KL
dp_kl = dp_df.pivot(index='feature', columns='level', values='kl').reindex(FEAT_ORDER)
clip_kl = clip_df.set_index('feature')['kl'].reindex(FEAT_ORDER)
dp_kl.insert(0, 'clip\n(ε→∞)', clip_kl.values)
dp_kl.columns = ['clip\n(ε→∞)'] + [f'ε={e}' for e in DP_EPSILONS]
make_heatmap(dp_kl,
             'KL D(P‖Q) (nats) — Differential privacy\n'
             '(P = JSD plateau; ≈23 marker = KL saturated)',
             'Greens', 0, KL_SAT,
             FIG_DIR / 'heatmap_kl_dp.png',
             annot_plateau=plateau_mask,
             annot_saturated=True, cbar_label='KL (nats)')

## 7. DP ε-curve

In [ ]:
cats_present = [c for c in CATEGORIES if c in metrics['category'].unique()]
n_cats = len(cats_present)
ncols = 3
nrows_plot = int(np.ceil(n_cats / ncols))

fig, axes = plt.subplots(nrows_plot, ncols,
                          figsize=(5 * ncols, 3.5 * nrows_plot))
axes_flat = axes.flat

for ax, cat in zip(axes_flat, cats_present):
    cat_dp = metrics[(metrics['technique'] == 'dp') &
                     (metrics['category'] == cat)]
    eps_vals = sorted(cat_dp['level'].unique())

    mean_jsd  = cat_dp.groupby('level')['jsd'].mean()
    mean_w1   = cat_dp.groupby('level')['wasserstein_norm'].mean()
    mean_pred = cat_dp.groupby('level')['w1_predicted'].mean()

    color_jsd  = CAT_COLORS.get(cat, '#333333')

    ax2 = ax.twinx()

    # JSD on primary axis
    ax.plot(eps_vals, mean_jsd.values, 'o-',
            color=color_jsd, linewidth=1.8, markersize=5,
            label='JSD (primary)')
    ax.set_ylim(-0.02, 1.05)
    ax.set_ylabel('Mean JSD', color=color_jsd, fontsize=8)
    ax.tick_params(axis='y', labelcolor=color_jsd)

    # W1 observed and predicted on secondary axis
    ax2.plot(eps_vals, mean_w1.values,   's--', color='#444',
             linewidth=1.2, markersize=4, label='W1 observed')
    ax2.plot(eps_vals, mean_pred.values, 'x:',  color='#999',
             linewidth=1.2, markersize=5, label='W1 predicted Δf/(ε·σ)')
    ax2.set_ylabel('Mean W1 / σ', color='#555', fontsize=7)
    ax2.tick_params(axis='y', labelcolor='#555', labelsize=7)

    ax.set_xscale('log')
    ax.set_xticks(eps_vals)
    ax.get_xaxis().set_major_formatter(mticker.ScalarFormatter())
    ax.set_xlabel('ε (log scale)')
    ax.set_title(f'{cat}', fontsize=9)

    # Combined legend
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2,
              fontsize=6, loc='upper right', framealpha=0.7)

# Hide unused axes
for ax in list(axes_flat)[n_cats:]:
    ax.set_visible(False)

fig.suptitle('DP ε-curve: mean JSD (primary) and W1 observed vs predicted (secondary)\n'
             'W1 observed ≈ predicted → W1 is design-predictable (REV-W1primary)',
             fontsize=9)
plt.tight_layout()
path = FIG_DIR / 'dp_epsilon_curve.png'
fig.savefig(path, dpi=DPI, bbox_inches='tight')
plt.show()
print(f'Saved: {path.name}')

## 8. Spearman ρ summary

In [ ]:
metrics_order = ['jsd', 'wasserstein_norm', 'kl']
metric_labels = {'jsd': 'JSD', 'wasserstein_norm': 'W1', 'kl': 'KL'}

fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=False)

for ax, metric in zip(axes, metrics_order):
    sub = spearman[spearman['metric'] == metric].copy()
    sub['label'] = sub.apply(
        lambda r: (
            f"noise α={r['level']:.1f}" if r['technique'] == 'noise'
            else f"supp {int(r['level']*100)}%" if r['technique'] == 'suppression'
            else f"DP ε={r['level']}" if r['technique'] == 'dp'
            else 'DP clip'
        ), axis=1
    )

    colors = []
    hatches = []
    for _, row in sub.iterrows():
        if row['technique'] == 'noise':
            colors.append('#4C72B0'); hatches.append('')
        elif row['technique'] == 'suppression':
            colors.append('#C44E52'); hatches.append('//')
        else:
            colors.append('#55A868'); hatches.append('xx')

    bars = ax.bar(sub['label'], sub['rho'],
                  color=colors, edgecolor='black', linewidth=0.5)
    for bar, hatch in zip(bars, hatches):
        bar.set_hatch(hatch)

    # Significance stars
    for i, (_, row) in enumerate(sub.iterrows()):
        if row['pvalue'] < 0.05:
            y = row['rho'] + (0.03 if row['rho'] >= 0 else -0.07)
            ax.text(i, y, '*', ha='center', va='bottom', fontsize=10)

    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_title(metric_labels[metric])
    ax.set_ylabel('Spearman ρ')
    ax.set_ylim(-1.05, 1.05)
    ax.set_xticklabels(sub['label'], rotation=45, ha='right', fontsize=7)

# Legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#4C72B0', label='Noise'),
    Patch(facecolor='#C44E52', hatch='//', label='Suppression (design-forced)'),
    Patch(facecolor='#55A868', hatch='xx', label='DP (design-forced)'),
]
fig.legend(handles=legend_elements, loc='lower center',
           ncol=3, fontsize=8, bbox_to_anchor=(0.5, -0.02),
           frameon=False)
fig.suptitle('Spearman ρ between distortion scores and Gini importance\n'
             '(* = p < 0.05; hatched = design-forced)',
             fontsize=9)
plt.tight_layout()
path = FIG_DIR / 'spearman_rho.png'
fig.savefig(path, dpi=DPI, bbox_inches='tight')
plt.show()
print(f'Saved: {path.name}')

## 9. Category-level JSD aggregation

In [ ]:
THRESHOLD = 0.1  # 'noticeable distortion'

arm_variants = [
    ('Noise α=0.1',    'noise_alpha0.1'),
    ('Noise α=0.5',    'noise_alpha0.5'),
    ('Suppression 50%','suppression_0.50'),
    ('DP ε=1.0',       'dp_eps1.0'),
    ('DP ε=10.0',      'dp_eps10.0'),
]

fig, axes = plt.subplots(1, len(arm_variants),
                          figsize=(4 * len(arm_variants), 5),
                          sharey=False)

for ax, (label, variant) in zip(axes, arm_variants):
    sub = metrics[metrics['variant'] == variant]
    cat_mean = sub.groupby('category')['jsd'].mean().reindex(CATEGORIES).fillna(0)
    cat_max  = sub.groupby('category')['jsd'].max().reindex(CATEGORIES).fillna(0)
    cat_frac = sub.groupby('category').apply(
        lambda g: (g['jsd'] > THRESHOLD).mean()
    ).reindex(CATEGORIES).fillna(0)

    x = np.arange(len(CATEGORIES))
    w = 0.25
    colors_list = [CAT_COLORS.get(c, '#999') for c in CATEGORIES]

    ax.bar(x - w, cat_mean.values,  w, label='Mean JSD',       color=colors_list, alpha=0.9)
    ax.bar(x,     cat_max.values,   w, label='Max JSD',        color=colors_list, alpha=0.55,
           edgecolor='black', linewidth=0.4)
    ax.bar(x + w, cat_frac.values,  w, label=f'Frac JSD>{THRESHOLD}',
           color=colors_list, alpha=0.35, hatch='//')

    ax.set_xticks(x)
    ax.set_xticklabels([c.replace(' ', '\n') for c in CATEGORIES],
                       fontsize=7)
    ax.set_ylim(0, 1.05)
    ax.set_ylabel('JSD / fraction')
    ax.set_title(label, fontsize=9)
    ax.axhline(THRESHOLD, color='grey', linestyle=':', linewidth=0.8)

# Shared legend on first axis
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='grey', alpha=0.9, label='Mean JSD'),
    Patch(facecolor='grey', alpha=0.55, edgecolor='black', label='Max JSD'),
    Patch(facecolor='grey', alpha=0.35, hatch='//', label=f'Frac > {THRESHOLD}'),
]
axes[0].legend(handles=legend_elements, fontsize=7, frameon=False)
fig.suptitle(f'JSD by feature category (dotted line = threshold {THRESHOLD})',
             fontsize=9)
plt.tight_layout()
path = FIG_DIR / 'category_jsd_summary.png'
fig.savefig(path, dpi=DPI, bbox_inches='tight')
plt.show()
print(f'Saved: {path.name}')

## 10. DP clipping-loss decomposition

In [ ]:
# Top 20 features by total JSD at ε=0.01
dp01 = metrics[metrics['variant'] == 'dp_eps0.01'].copy()
top20 = dp01.nlargest(20, 'jsd')['feature'].tolist()

dp_top = metrics[
    (metrics['technique'] == 'dp') & (metrics['feature'].isin(top20))
].copy()

eps_vals = sorted(dp_top['level'].unique())

fig, axes = plt.subplots(1, len(eps_vals),
                          figsize=(2.5 * len(eps_vals), 6),
                          sharey=True)

for ax, eps in zip(axes, eps_vals):
    sub = dp_top[dp_top['level'] == eps].set_index('feature').reindex(top20)
    clip_vals  = sub['jsd_clip_component'].fillna(0).values
    noise_vals = sub['jsd_noise_component'].fillna(0).values
    cat_colors_t20 = [CAT_COLORS.get(
        meta[meta['feature'] == f]['category'].values[0]
        if f in meta['feature'].values else 'Unknown', '#999')
        for f in top20]

    y = np.arange(len(top20))
    ax.barh(y, clip_vals,  0.6, color='#aaaaaa', label='Clipping (ε-constant)')
    ax.barh(y, noise_vals, 0.6, left=clip_vals,
            color=[CAT_COLORS.get(
                meta[meta['feature'] == f]['category'].values[0]
                if f in meta['feature'].values else '', '#555')
                for f in top20],
            alpha=0.85, label='Noise component')

    ax.set_yticks(y)
    if ax is axes[0]:
        ax.set_yticklabels([f[:22] for f in top20], fontsize=6.5)
    ax.set_xlabel('JSD')
    ax.set_xlim(0, 1.05)
    ax.set_title(f'ε={eps}', fontsize=9)
    ax.axvline(0.1, color='grey', linestyle=':', linewidth=0.7)

axes[0].legend(fontsize=7, loc='lower right', frameon=False)
fig.suptitle('DP JSD decomposition: clipping (grey) + noise (coloured) — top 20 features by JSD at ε=0.01',
             fontsize=9)
plt.tight_layout()
path = FIG_DIR / 'dp_clip_decomposition.png'
fig.savefig(path, dpi=DPI, bbox_inches='tight')
plt.show()
print(f'Saved: {path.name}')

## 11. Summary of output files

In [ ]:
import os
print('Figures saved to:', FIG_DIR)
print()
for f in sorted(FIG_DIR.glob('*.png')):
    size_kb = os.path.getsize(f) / 1024
    print(f'  {f.name:<40}  {size_kb:.0f} KB')

## 12. Feature-metrics table

Exports `results/feature_metrics_table.csv` — one row per feature, one column per (technique × level × metric).  
Useful as an appendix data table in the report.  
Also renders a formatted summary: **top 15 features by mean JSD across all variants**.

In [ ]:
# ── Build wide table: feature × (variant × metric) ────────────────────────────
METRIC_COLS = ['jsd', 'wasserstein_norm', 'kl']
METRIC_LABELS = {'jsd': 'JSD', 'wasserstein_norm': 'W1', 'kl': 'KL'}

# Human-readable column name for each (variant, metric)
def col_name(variant, metric):
    label = METRIC_LABELS[metric]
    if 'noise' in variant:
        alpha = variant.split('alpha')[1]
        return f'noise_α{alpha}_{label}'
    elif 'suppression' in variant:
        pct = int(float(variant.split('_')[1]) * 100)
        return f'supp_{pct}%_{label}'
    elif 'clip_only' in variant:
        return f'dp_clip_{label}'
    else:
        eps = variant.split('eps')[1]
        return f'dp_ε{eps}_{label}'

# Pivot to wide format
wide_frames = []
for variant in sorted(metrics['variant'].unique()):
    sub = metrics[metrics['variant'] == variant].set_index('feature')
    for metric in METRIC_COLS:
        col = col_name(variant, metric)
        wide_frames.append(sub[metric].rename(col))

wide = pd.concat(wide_frames, axis=1)

# Add feature metadata
wide.insert(0, 'category', meta.set_index('feature')['category'].reindex(wide.index))
wide.insert(1, 'gini_importance',
            gini.set_index('feature')['importance'].reindex(wide.index).round(6))

# Sort by Gini importance descending
wide = wide.sort_values('gini_importance', ascending=False)

# Save full table
out_path = RESULTS / 'feature_metrics_table.csv'
wide.to_csv(out_path)
print(f'Saved: {out_path.name}  ({wide.shape[0]} features × {wide.shape[1]} columns)')
print(f'Columns: {list(wide.columns[:6])} … ({len(wide.columns)} total)')


In [ ]:
# ── Top-15 features by mean JSD across all techniques ────────────────────────
jsd_cols = [c for c in wide.columns if c.endswith('_JSD')]
wide['mean_jsd_all'] = wide[jsd_cols].mean(axis=1)
top15 = wide.nlargest(15, 'mean_jsd_all')

# Select representative columns for display
display_cols = (
    ['category', 'gini_importance']
    + [c for c in wide.columns if c.endswith('_JSD') and
       any(x in c for x in ['noise_α0.5', 'supp_50%', 'dp_ε1.0', 'dp_ε10.0'])]
    + ['mean_jsd_all']
)
display_cols = [c for c in display_cols if c in top15.columns]

summary = top15[display_cols].copy()
# Round for readability
for col in summary.select_dtypes(include='number').columns:
    summary[col] = summary[col].round(4)

print('Top 15 features by mean JSD across all variants:')
print(summary.to_string())

# Also save the top-15 as a separate CSV for easy copy-paste into the report
top15_path = RESULTS / 'top15_features_by_jsd.csv'
top15[display_cols].round(4).to_csv(top15_path)
print(f'\nSaved: {top15_path.name}')
